# Hands-On LangChain #5: Information Retrieval

Companion to [the full tutorial](https://todatabeyond.com/blog/hands-on-langchain-for-llm-applications-development-information-retrieval).

The default path uses real pretrained MiniLM embeddings and Chroma for similarity search, MMR, metadata filtering, and persistence. It downloads the model on first use and needs no API key. Self-query and compression use explicitly scripted `FakeListLLM` responses to test LangChain parsing, filter translation, and extraction plumbing; these are NOT claims of language-model reasoning or live OpenAI execution. Local lecture snippets are fixtures, not original CS229 PDFs. The final opt-in appendix contains every updated original tutorial code step with hosted embeddings and LLM inference.

## 1. Install dependencies

In [ ]:
!pip install -q "langchain-chroma>=1.0,<2" "langchain-classic>=1.0,<2" "langchain-community>=0.4,<0.5" "langchain-openai>=1.0,<2" "langchain-text-splitters>=1.0,<2" lark pypdf numpy python-dotenv

## 2. Pretrained local embeddings

In [ ]:
from pathlib import Path
import tempfile
import numpy as np
from langchain_core.embeddings import Embeddings
from chromadb.utils.embedding_functions import ONNXMiniLM_L6_V2

class LocalMiniLMEmbeddings(Embeddings):
    def __init__(self):
        self.model = ONNXMiniLM_L6_V2(preferred_providers=["CPUExecutionProvider"])
        self.model.DOWNLOAD_PATH = Path(tempfile.gettempdir()) / "tdb-minilm-model-cache"

    def embed_documents(self, texts: list[str]) -> list[list[float]]:
        return [vector.tolist() for vector in self.model(texts)]

    def embed_query(self, text: str) -> list[float]:
        return self.embed_documents([text])[0]

embedding = LocalMiniLMEmbeddings()
print(f"Embedding dimensions: {len(embedding.embed_query('example'))}")

## 3. Similarity search versus MMR
These are the original three mushroom example sentences. This is a retrieval demonstration, not mushroom-identification or safety advice. Selection can depend on the embedding model and MMR balance.

In [ ]:
from langchain_chroma import Chroma
from langchain_core.documents import Document

persist_directory = tempfile.mkdtemp(prefix="tdb-retrieval-")
texts = [
    "The Amanita phalloides has a large and imposing epigeous (aboveground) fruiting body (basidiocarp).",
    "A mushroom with a large fruiting body is the Amanita phalloides. Some varieties are all-white.",
    "A. phalloides, a.k.a Death Cap, is one of the most poisonous of all known mushrooms.",
]
smalldb = Chroma.from_texts(texts, embedding=embedding, collection_name="mushrooms", persist_directory=persist_directory)
question = "Tell me about all-white mushrooms with large fruiting bodies"
similar = smalldb.similarity_search(question, k=2)
diverse = smalldb.max_marginal_relevance_search(question, k=2, fetch_k=3, lambda_mult=0.2)
assert len(similar) == len(diverse) == 2
assert len({d.page_content for d in diverse}) == 2
assert any("poisonous" in d.page_content for d in diverse)
print("Similarity search:")
for doc in similar: print(doc.page_content)
print("MMR (lambda_mult=0.2):")
for doc in diverse: print(doc.page_content)

## 4. Duplicate lecture passages and metadata filtering

In [ ]:
lecture1 = "docs/cs229_lectures/MachineLearning-Lecture01.pdf"
lecture2 = "docs/cs229_lectures/MachineLearning-Lecture02.pdf"
lecture3 = "docs/cs229_lectures/MachineLearning-Lecture03.pdf"
matlab = "MATLAB makes it easy to write numerical routines using matrices and plot data. The course meets on weekdays and has a final project."
tutorial = "We will have a short MATLAB tutorial in a discussion section. Study groups can discuss homework after class."
fixtures = [
    Document(page_content=matlab, metadata={"source": lecture1, "page": 8}),
    Document(page_content=matlab, metadata={"source": lecture1, "page": 8}),
    Document(page_content=tutorial, metadata={"source": lecture1, "page": 9}),
    Document(page_content="Regression in the third lecture minimizes squared prediction errors. Students should read the course schedule.", metadata={"source": lecture3, "page": 0}),
    Document(page_content="Regression in the second lecture introduces supervised learning.", metadata={"source": lecture2, "page": 1}),
    Document(page_content="The class discusses clustering and unsupervised learning.", metadata={"source": lecture2, "page": 3}),
]
vectordb = Chroma.from_documents(fixtures, embedding=embedding, collection_name="lectures", persist_directory=persist_directory)
question = "what did they say about matlab?"
docs_ss = vectordb.similarity_search(question, k=3)
docs_mmr = vectordb.max_marginal_relevance_search(question, k=3, fetch_k=6, lambda_mult=0.5)
assert len({d.page_content for d in docs_ss}) < len(docs_ss)
assert len({d.page_content for d in docs_mmr}) == 3
filtered = vectordb.similarity_search("regression", k=3, filter={"source": lecture3})
assert filtered and all(d.metadata["source"] == lecture3 for d in filtered)
print(f"Lecture passages: {len(vectordb.get()['ids'])}")
print(f"Similarity-search unique passages: {len({d.page_content for d in docs_ss})} / 3")
print(f"MMR unique passages: {len({d.page_content for d in docs_mmr})} / 3")
print(f"Metadata-filtered passages: {len(filtered)} (lecture 3 only)")

## 5. Self-query plumbing with a scripted model

`FakeListLLM` returns a fixed structured query. The real query-constructor parser, Chroma translator, metadata filter, and vector lookup run normally. This verifies wiring, not that an LLM inferred the right filter. The hosted appendix replaces the stub with `ChatOpenAI`.

In [ ]:
from langchain_core.language_models.fake import FakeListLLM
from langchain_classic.retrievers.self_query.base import SelfQueryRetriever
from langchain_classic.chains.query_constructor.base import AttributeInfo
from langchain_community.query_constructors.chroma import ChromaTranslator
import json

metadata_field_info = [
    AttributeInfo(name="source", description="The source lecture PDF path", type="string"),
    AttributeInfo(name="page", description="Zero-based PDF page number", type="integer"),
]
scripted_query = json.dumps({"query": "regression", "filter": f'eq("source", "{lecture3}")'})
query_llm = FakeListLLM(responses=[scripted_query])
self_query = SelfQueryRetriever.from_llm(
    query_llm, vectordb, "Lecture notes", metadata_field_info,
    structured_query_translator=ChromaTranslator(), search_kwargs={"k": 3},
)
self_docs = self_query.invoke("What did they say about regression in the third lecture?")
assert self_docs and all(d.metadata["source"] == lecture3 for d in self_docs)
print(f"Scripted self-query parser/filter verified: {len(self_docs)} passage from lecture 3")

## 6. Compression and MMR composition with scripted extraction

The extraction responses below are explicit test fixtures, matched to the actual retrieval order. `LLMChainExtractor` and `ContextualCompressionRetriever` execute normally, including dropping `NO_OUTPUT`, preserving metadata, and returning shorter passages. The script does not substitute for testing a real LLM's extraction quality.

In [ ]:
from langchain_classic.retrievers import ContextualCompressionRetriever
from langchain_classic.retrievers.document_compressors import LLMChainExtractor

def fixture_compression(search_type):
    base = vectordb.as_retriever(search_type=search_type, search_kwargs={"k": 3, **({"fetch_k": 6, "lambda_mult": 0.5} if search_type == "mmr" else {})})
    retrieved = base.invoke("what did they say about matlab?")
    responses = [
        d.page_content.split(". ")[0] + "." if "MATLAB" in d.page_content else "NO_OUTPUT"
        for d in retrieved
    ]
    extractor = LLMChainExtractor.from_llm(FakeListLLM(responses=responses))
    retriever = ContextualCompressionRetriever(base_compressor=extractor, base_retriever=base)
    return retrieved, retriever.invoke("what did they say about matlab?")

raw_ss, compressed_ss = fixture_compression("similarity")
raw_mmr, compressed_mmr = fixture_compression("mmr")
assert compressed_ss and compressed_mmr
assert len({d.page_content for d in compressed_ss}) < len(compressed_ss)
assert len({d.page_content for d in compressed_mmr}) == len(compressed_mmr)
assert sum(len(d.page_content) for d in compressed_ss) < sum(len(d.page_content) for d in raw_ss)
assert all(any(d.page_content in original.page_content and d.metadata == original.metadata for original in raw_mmr) for d in compressed_mmr)
print(f"Similarity + scripted compression: {len(compressed_ss)} excerpts, {len({d.page_content for d in compressed_ss})} unique")
print(f"MMR + scripted compression: {len(compressed_mmr)} excerpts, {len({d.page_content for d in compressed_mmr})} unique")
for doc in compressed_mmr: print(doc.page_content)

## 7. Persistence and verification summary

In [ ]:
reopened = Chroma(collection_name="lectures", embedding_function=embedding, persist_directory=persist_directory)
assert len(reopened.get()["ids"]) == len(fixtures)
print("Verified embedding dimensions: 384")
print("Verified similarity search and MMR: original mushroom example")
print("Verified duplicate diversification: 2 unique -> 3 unique passages")
print("Verified metadata filtering: lecture 3 only")
print("Verified scripted self-query parsing and filter translation")
print("Verified scripted compression and MMR composition")
print("Verified Chroma persistence: 6 passages after reopening")
print("All local information-retrieval assertions passed.")

## 8. Complete optional hosted workflow

The next cells contain all updated original tutorial code steps. Enable them only after setting `OPENAI_API_KEY` and `OPENAI_CHAT_MODEL` locally and placing the three CS229 PDFs in `docs/cs229_lectures/`. They build a fresh index with the selected embedding model, avoiding incompatible prior embeddings. Calls can incur charges. This appendix was syntax-checked, but hosted inference was not execution-tested; archived output is not asserted for current models. MMR encourages diversity but does not guarantee deduplication, and LLM filters/extractions should be inspected before trusting them.

In [ ]:
RUN_HOSTED_EXAMPLE = False

In [ ]:
if RUN_HOSTED_EXAMPLE:
    import os
    from pathlib import Path
    from tempfile import mkdtemp

In [ ]:
if RUN_HOSTED_EXAMPLE:
    from dotenv import load_dotenv, find_dotenv
    _ = load_dotenv(find_dotenv()) # read local .env file

In [ ]:
if RUN_HOSTED_EXAMPLE:
    for name in ("OPENAI_API_KEY", "OPENAI_CHAT_MODEL"):
        if not os.environ.get(name):
            raise RuntimeError(f"Set {name} locally before running hosted examples.")

In [ ]:
if RUN_HOSTED_EXAMPLE:
    from langchain_chroma import Chroma
    from langchain_openai import OpenAIEmbeddings
    persist_directory = mkdtemp(prefix="langchain-retrieval-chroma-")

In [ ]:
if RUN_HOSTED_EXAMPLE:
    from langchain_community.document_loaders import PyPDFLoader
    from langchain_text_splitters import RecursiveCharacterTextSplitter
    
    # Build a fresh index with this model, rather than reopen incompatible embeddings.
    pdf_paths = [f"docs/cs229_lectures/MachineLearning-Lecture{i:02d}.pdf" for i in (1, 1, 2, 3)]
    if not all(Path(path).is_file() for path in pdf_paths):
        raise FileNotFoundError("Place all three CS229 lecture PDFs under docs/cs229_lectures/.")
    lecture_docs = []
    for path in pdf_paths:
        lecture_docs.extend(PyPDFLoader(path).load())
    splits = RecursiveCharacterTextSplitter(chunk_size=1500, chunk_overlap=150).split_documents(lecture_docs)
    embedding = OpenAIEmbeddings(model="text-embedding-3-small")
    vectordb = Chroma.from_documents(
        splits, embedding=embedding, collection_name="lectures",
        persist_directory=persist_directory,
    )
    print(len(vectordb.get()["ids"]))

In [ ]:
if RUN_HOSTED_EXAMPLE:
    texts = [
        """The Amanita phalloides has a large and imposing epigeous (aboveground) fruiting body (basidiocarp).""",
        """A mushroom with a large fruiting body is the Amanita phalloides. Some varieties are all-white.""",
        """A. phalloides, a.k.a Death Cap, is one of the most poisonous of all known mushrooms.""",
    ]
    smalldb = Chroma.from_texts(texts, embedding=embedding, collection_name="mushrooms", persist_directory=persist_directory)
    question = "Tell me about all-white mushrooms with large fruiting bodies"

In [ ]:
if RUN_HOSTED_EXAMPLE:
    print(smalldb.similarity_search(question, k=2))

In [ ]:
if RUN_HOSTED_EXAMPLE:
    print(smalldb.max_marginal_relevance_search(question,k=2, fetch_k=3))

In [ ]:
if RUN_HOSTED_EXAMPLE:
    question = "what did they say about matlab?"
    docs_ss = vectordb.similarity_search(question,k=3)
    print(docs_ss[0].page_content[:100])

In [ ]:
if RUN_HOSTED_EXAMPLE:
    print(docs_ss[1].page_content[:100])

In [ ]:
if RUN_HOSTED_EXAMPLE:
    docs_mmr = vectordb.max_marginal_relevance_search(question,k=3)
    print(docs_mmr[0].page_content[:100])

In [ ]:
if RUN_HOSTED_EXAMPLE:
    print(docs_mmr[1].page_content[:100])

In [ ]:
if RUN_HOSTED_EXAMPLE:
    question = "what did they say about regression in the third lecture?"
    docs = vectordb.similarity_search(
        question,
        k=3,
        filter={"source":"docs/cs229_lectures/MachineLearning-Lecture03.pdf"}
    )
    
    for d in docs:
        print(d.metadata)

In [ ]:
if RUN_HOSTED_EXAMPLE:
    from langchain_openai import ChatOpenAI
    from langchain_community.query_constructors.chroma import ChromaTranslator
    from langchain_classic.retrievers.self_query.base import SelfQueryRetriever
    from langchain_classic.chains.query_constructor.base import AttributeInfo

In [ ]:
if RUN_HOSTED_EXAMPLE:
    metadata_field_info = [
        AttributeInfo(
            name="source",
            description="The lecture the chunk is from, should be one of `docs/cs229_lectures/MachineLearning-Lecture01.pdf`, `docs/cs229_lectures/MachineLearning-Lecture02.pdf`, or `docs/cs229_lectures/MachineLearning-Lecture03.pdf`",
            type="string",
        ),
        AttributeInfo(
            name="page",
            description="The page from the lecture",
            type="integer",
        ),
    ]

In [ ]:
if RUN_HOSTED_EXAMPLE:
    document_content_description = "Lecture notes"
    llm = ChatOpenAI(model=os.environ["OPENAI_CHAT_MODEL"], temperature=0)
    retriever = SelfQueryRetriever.from_llm(
        llm,
        vectordb,
        document_content_description,
        metadata_field_info,
        structured_query_translator=ChromaTranslator(),
        verbose=True
    )
    
    question = "what did they say about regression in the third lecture?"
    # Invoke the retriever: the archive omitted this step and reused old docs.
    docs = retriever.invoke(question)

In [ ]:
if RUN_HOSTED_EXAMPLE:
    for d in docs:
        print(d.metadata)

In [ ]:
if RUN_HOSTED_EXAMPLE:
    from langchain_classic.retrievers import ContextualCompressionRetriever
    from langchain_classic.retrievers.document_compressors import LLMChainExtractor

In [ ]:
if RUN_HOSTED_EXAMPLE:
    def pretty_print_docs(docs):
        print(f"\n{'-' * 100}\n".join([f"Document {i+1}:\n\n" + d.page_content for i, d in enumerate(docs)]))

In [ ]:
if RUN_HOSTED_EXAMPLE:
    # Wrap our vectorstore
    llm = ChatOpenAI(model=os.environ["OPENAI_CHAT_MODEL"], temperature=0)
    compressor = LLMChainExtractor.from_llm(llm)
    
    compression_retriever = ContextualCompressionRetriever(
        base_compressor=compressor,
        base_retriever=vectordb.as_retriever()
    )

In [ ]:
if RUN_HOSTED_EXAMPLE:
    question = "what did they say about matlab?"
    compressed_docs = compression_retriever.invoke(question)
    print(pretty_print_docs(compressed_docs))

In [ ]:
if RUN_HOSTED_EXAMPLE:
    compression_retriever = ContextualCompressionRetriever(
        base_compressor=compressor,
        base_retriever=vectordb.as_retriever(search_type = "mmr")
    )

In [ ]:
if RUN_HOSTED_EXAMPLE:
    question = "what did they say about matlab?"
    compressed_docs = compression_retriever.invoke(question)
    print(pretty_print_docs(compressed_docs))